In [82]:
import cv2
from glob import glob
import pathlib
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

In [83]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Flatten,Dense,Conv3D,MaxPool3D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy

In [84]:
fish=pathlib.Path("/content/drive/MyDrive/Datasets/Sea CNN")

In [85]:
a=list(fish.glob("Clams/*.jpg"))
b=list(fish.glob('Jelly fish/*.jpg'))
c=list(fish.glob('Octopus/*.jpg'))
d=list(fish.glob('Penguin/*.jpg'))
e=list(fish.glob('Sharks/*.jpg'))

In [86]:
len(a),len(b),len(c),len(d),len(e)

(99, 0, 100, 100, 98)

In [87]:
fish_dict={"Clams":a,
              "Jelly Fish":b,
              "Octopus":c,
              "Penguin":d,
              "Sharks":e}
fish_class={"Clams":0,
              "Jelly Fish":1,
              "Octopus":2,
              "Penguin":3,
              "Sharks":4}

In [88]:
x=[]
y=[]

In [89]:
for i in fish_dict:
  fish_name=i
  fish_path_list=fish_dict[fish_name]
  for path in fish_path_list:
    img=cv2.imread(str(path))
    img=cv2.resize(img,(224,224))
    img=img/255
    x.append(img)
    cls=fish_class[fish_name]
    y.append(cls)

In [90]:
len(x)

397

In [91]:
x=np.array(x)
y=np.array(y)

In [92]:
from sklearn.model_selection import train_test_split

In [93]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.33, random_state=42)

In [94]:
len(X_train),len(y_train),len(X_test),len(y_test)

(265, 265, 132, 132)

In [95]:
X_train.shape

(265, 224, 224, 3)

In [96]:
X_train.shape,X_test.shape

((265, 224, 224, 3), (132, 224, 224, 3))

In [97]:
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2


In [98]:
base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3),
                                               include_top=False, #remove layers after flatten
                                               weights='imagenet') #inbuild dataset images

In [99]:
print("[INFO] summary for base model...")
print(base_model.summary())

[INFO] summary for base model...
Model: "mobilenetv2_1.00_224"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_4 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 Conv1 (Conv2D)              (None, 112, 112, 32)         864       ['input_4[0][0]']             
                                                                                                  
 bn_Conv1 (BatchNormalizati  (None, 112, 112, 32)         128       ['Conv1[0][0]']               
 on)                                                                                              
                                                                                                  
 Conv1_relu (ReLU)           (None, 112, 112, 

In [100]:
from tensorflow.keras.layers import MaxPooling2D
from keras.layers import Dropout
from tensorflow.keras.models import Model
headModel = base_model.output
headModel = MaxPooling2D(pool_size=(2, 2))(headModel)
headModel = Flatten(name="flatten")(headModel)
headModel = Dense(32, activation="relu")(headModel)
headModel = Dropout(0.2)(headModel)
headModel = Dense(5, activation="softmax")(headModel)
# place the head FC model on top of the base model (this will become
# the actual model we will train)
model = Model(inputs=base_model.input, outputs=headModel)
# loop over all layers in the base model and freeze them so they will
# *not* be updated during the first training process
for layer in base_model.layers:
	layer.trainable = False

In [104]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
# compile our model (this needs to be done after our setting our
# layers to being non-trainable)
print("[INFO] compiling model...")
opt = Adam()
model.compile(loss="sparse_categorical_crossentropy", optimizer=opt,metrics=["accuracy"])
print("[INFO] training head...")
callback=EarlyStopping(monitor='val_accuracy',patience=20)
model_hist=model.fit(X_train,y_train,epochs=100,validation_data=(X_test,y_test),batch_size=180)

[INFO] compiling model...
[INFO] training head...
Epoch 1/100
2/2 [==============================] - 6s 2s/step - loss: 3.9410 - accuracy: 0.3019 - val_loss: 3.6107 - val_accuracy: 0.2955
Epoch 2/100
2/2 [==============================] - 1s 480ms/step - loss: 2.2452 - accuracy: 0.4868 - val_loss: 0.8256 - val_accuracy: 0.6364
Epoch 3/100
2/2 [==============================] - 1s 483ms/step - loss: 0.9104 - accuracy: 0.5774 - val_loss: 0.7421 - val_accuracy: 0.6515
Epoch 4/100
2/2 [==============================] - 1s 436ms/step - loss: 0.7503 - accuracy: 0.6075 - val_loss: 0.6588 - val_accuracy: 0.7045
Epoch 5/100
2/2 [==============================] - 1s 434ms/step - loss: 0.5908 - accuracy: 0.7245 - val_loss: 0.6945 - val_accuracy: 0.6818
Epoch 6/100
2/2 [==============================] - 1s 327ms/step - loss: 0.5559 - accuracy: 0.7321 - val_loss: 0.8113 - val_accuracy: 0.6667
Epoch 7/100
2/2 [==============================] - 1s 323ms/step - loss: 0.5668 - accuracy: 0.7170 - val_lo

In [105]:
model.save("weights.h5")

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [106]:
from tensorflow.keras.preprocessing import image
# testing the model

def testing_image(image_directory):
  test_image=image.load_img(image_directory,target_size=(224,224))
  test_image=image.img_to_array(test_image)
  test_image=np.expand_dims(test_image,axis=0)
  test_image=test_image/255
  result=model.predict(x=test_image)
  print(result)
  if np.argmax(result)==0:
    prediction='Clams'
  elif np.argmax(result)==1:
    prediction='Jelly Fish'
  elif np.argmax(result)==2:
    prediction='Octopus'
  elif np.argmax(result)==3:
    prediction='Penguin'
  else:
    prediction="Sharks"
  return prediction

In [107]:
print(testing_image('/content/drive/MyDrive/Datasets/Sea CNN/Clams/15663710458_e8f7bb2742_o.jpg'))

1/1 [==============================] - 1s 1s/step
[[9.9910420e-01 1.0677690e-05 8.8488922e-04 9.1097724e-10 1.2761217e-07]]
Clams


In [108]:
print(testing_image("/content/drive/MyDrive/Datasets/Sea CNN/Octopus/25281922820_87d1275dfd_b.jpg"))

1/1 [==============================] - 0s 26ms/step
[[4.9665368e-01 7.6755555e-07 5.0332057e-01 1.3773392e-11 2.4973695e-05]]
Octopus


In [109]:
print(testing_image('/content/drive/MyDrive/Datasets/Sea CNN/Penguin/14968911465_cf7d58c99a_o.jpg'))

1/1 [==============================] - 0s 24ms/step
[[6.8612059e-07 7.7599305e-10 2.0704184e-07 9.3201506e-01 6.7984089e-02]]
Penguin
